In [52]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu

# Partie A:  Audit et nettoyage

*Exploration*

In [54]:
df = pd.read_csv('../data/points_eau.csv')
print(f"\n shape : {df.shape} ")
print(f"\n columns : {df.columns.to_list() } ")
print(f"Manquants : {df.isnull().sum().sum()} valeurs / {df.isnull().mean().mean()*100:.1f}% du total")
print(f"Types     : \n{df.dtypes.value_counts()}")
print(f"info : {df.info()}")
print("\n description :", df.describe())


 shape : (11472, 32) 

 columns : ['id_point_eau', 'date_releve', 'departement', 'commune', 'latitude', 'longitude', 'altitude_m', 'annee_construction', 'type_ouvrage', 'type_pompe', 'profondeur_forage_m', 'niveau_statique_m', 'debit_essai_m3_h', 'qualite_eau', 'nb_menages', 'population_desservie', 'distance_village_m', 'nb_points_eau_village', 'mode_gestion', 'mode_paiement', 'cotisation_mensuelle_fcfa', 'technicien_forme_village', 'stock_pieces_rechange_commune', 'distance_atelier_km', 'maitre_ouvrage', 'installateur', 'nb_pannes_12_mois', 'mois_depuis_derniere_maintenance', 'nb_jours_arret_12_mois', 'intervention_prevue', 'cout_reparation_estime_fcfa', 'etat_fonctionnement'] 
Manquants : 5515 valeurs / 1.5% du total
Types     : 
str        13
float64    13
int64       6
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 11472 entries, 0 to 11471
Data columns (total 32 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                 

# A1. Tableau des valeurs manquantes

In [55]:
tableau = pd.DataFrame({
    "n_manquants": df.isna().sum(),
    "taux_%": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
})
tableau = tableau[tableau["n_manquants"] > 0].sort_values("taux_%", ascending=False)

print(f"Cellules vides   : {df.isna().sum().sum() / df.size * 100:.2f} % du tableau")
print(f"Lignes complètes : {df.notna().all(axis=1).mean()*100:.1f} %  "
      f"(supprimer les lignes trouées coûterait {(1-df.notna().all(axis=1).mean())*100:.0f} % du jeu)")
display(tableau)

Cellules vides   : 1.50 % du tableau
Lignes complètes : 60.3 %  (supprimer les lignes trouées coûterait 40 % du jeu)


,n_manquants,taux_%,dtype
niveau_statique_m,1679,14.64,float64
debit_essai_m3_h,1056,9.21,float64
mois_depuis_derniere_maintenance,896,7.81,float64
altitude_m,698,6.08,float64
nb_menages,480,4.18,float64
qualite_eau,406,3.54,str
cotisation_mensuelle_fcfa,300,2.62,float64


In [56]:
# COLS_CAT = [c for c in df.select_dtypes(include=["object", "string", "category"]).columns   if c != "id_point_eau"]
COLS_CAT = [c for c in df.select_dtypes(include=["object", "string", "category"]).columns
            if c not in ("id_point_eau", "etat_fonctionnement", "date_releve")]

COLS_NUM = df.select_dtypes(include=["number"]).columns.to_list()

print("Categorielles :", COLS_CAT)
print("Numeriques    :", COLS_NUM)

Categorielles : ['departement', 'commune', 'type_ouvrage', 'type_pompe', 'qualite_eau', 'mode_gestion', 'mode_paiement', 'maitre_ouvrage', 'installateur', 'intervention_prevue']
Numeriques    : ['latitude', 'longitude', 'altitude_m', 'annee_construction', 'profondeur_forage_m', 'niveau_statique_m', 'debit_essai_m3_h', 'nb_menages', 'population_desservie', 'distance_village_m', 'nb_points_eau_village', 'cotisation_mensuelle_fcfa', 'technicien_forme_village', 'stock_pieces_rechange_commune', 'distance_atelier_km', 'nb_pannes_12_mois', 'mois_depuis_derniere_maintenance', 'nb_jours_arret_12_mois', 'cout_reparation_estime_fcfa']


In [57]:
def diagnostic_manquants(df, col, cols_cat=COLS_CAT, cols_num=COLS_NUM, alpha=0.05):

    mask = df[col].isna()
    print("=" * 78)
    print(f" {col} - {mask.sum()} manquants ({mask.mean()*100:.1f} %)")
    print("=" * 78)

    lignes = []
    for c in cols_cat:
        if c == col:
            continue
        tab = pd.crosstab(df[c], mask)
        if tab.shape[0] < 2 or tab.shape[1] < 2:
            continue
        lignes.append({"variable": c, "type": "cat", "p_value": chi2_contingency(tab)[1]})

    for c in cols_num:
        if c == col:
            continue
        a = df.loc[~mask, c].dropna()
        b = df.loc[mask, c].dropna()
        if len(b) < 10:
            continue
        lignes.append({"variable": c, "type": "num", "p_value": mannwhitneyu(a, b).pvalue})

    res = pd.DataFrame(lignes).sort_values("p_value").reset_index(drop=True)
    n_tests = len(res)
    seuil_bonf = alpha / n_tests
    res["p_value"] = res["p_value"].round(4)
    res["brut"] = np.where(res["p_value"] < alpha, "LIE", "indep.")
    res["corrige"] = np.where(res["p_value"] < seuil_bonf, "LIE", "indep.")
    print(f"{n_tests} tests  seuil brut {alpha}, seuil Bonferroni {seuil_bonf:.4f}")
    print(res.to_string(index=False))

    med = df.groupby(mask)[[c for c in cols_num if c != col]].median().T
    med.columns = ["complet", "manquant"]
    med["ecart_%"] = ((med["manquant"] / med["complet"] - 1) * 100).round(1)
    print("\nMédianes comparées :")
    print(med.round(1).to_string())


    repartition = (pd.crosstab(mask, df["etat_fonctionnement"], normalize="index") * 100).round(1)
    repartition.index = ["complet", "manquant"]
    effectifs = mask.value_counts().reindex([False, True]).values

    p_cible = chi2_contingency(pd.crosstab(df["etat_fonctionnement"], mask))[1]
    print(f"\nRepartition de la cible (%)  n = {effectifs[0]} / {effectifs[1]} :")
    print(repartition.to_string())
    print(f"p = {p_cible:.4f} -> l'absence est "
          f"{'INFORMATIVE' if p_cible < alpha else 'non informative'} sur la cible\n")
    return res



In [58]:
resultats = {}
for col in tableau.index:
    resultats[col] = diagnostic_manquants(df, col)

 niveau_statique_m - 1679 manquants (14.6 %)
28 tests  seuil brut 0.05, seuil Bonferroni 0.0018
                        variable type  p_value   brut corrige
                      nb_menages  num   0.0172    LIE  indep.
            population_desservie  num   0.0206    LIE  indep.
                    installateur  cat   0.0208    LIE  indep.
             distance_atelier_km  num   0.0253    LIE  indep.
           nb_points_eau_village  num   0.0421    LIE  indep.
               nb_pannes_12_mois  num   0.1625 indep.  indep.
             intervention_prevue  cat   0.1843 indep.  indep.
                debit_essai_m3_h  num   0.2069 indep.  indep.
mois_depuis_derniere_maintenance  num   0.2134 indep.  indep.
        technicien_forme_village  num   0.2191 indep.  indep.
                         commune  cat   0.2244 indep.  indep.
     cout_reparation_estime_fcfa  num   0.2328 indep.  indep.
   stock_pieces_rechange_commune  num   0.2797 indep.  indep.
              distance_village_m  nu

# A2. Détection de doublons et traitement

In [59]:
n_doublons = df.duplicated().sum()
print(f"Doublons : {n_doublons}  " )

Doublons : 65  


In [60]:
n_doublons_id = df["id_point_eau"].duplicated().sum()
print(f"Doublons sur id point d'eau : {n_doublons_id}  " )

Doublons sur id point d'eau : 72  


In [61]:
doublons = df[df.duplicated(keep=False)].sort_values("id_point_eau")
print(f"Lignes impliquées : {len(doublons)}")
display(doublons.head(6))

Lignes impliquées : 130


,id_point_eau,date_releve,departement,commune,latitude,longitude,altitude_m,annee_construction,type_ouvrage,type_pompe,...,stock_pieces_rechange_commune,distance_atelier_km,maitre_ouvrage,installateur,nb_pannes_12_mois,mois_depuis_derniere_maintenance,nb_jours_arret_12_mois,intervention_prevue,cout_reparation_estime_fcfa,etat_fonctionnement
695,PE-000051,2025-11-20,Borgou,N'Dali,9.22114,2.80126,214.9,2021,Poste d'eau autonome,Immergee electrique,...,0,26.7,ONG internationale,ENT-008,0,7.0,12.0,Aucune,0.0,fonctionnel
4264,PE-000051,2025-11-20,Borgou,N'Dali,9.22114,2.80126,214.9,2021,Poste d'eau autonome,Immergee electrique,...,0,26.7,ONG internationale,ENT-008,0,7.0,12.0,Aucune,0.0,fonctionnel
8843,PE-000759,2025-11-02,Borgou,Parakou,9.91264,3.02302,NaN,1995,Forage equipe PMH,India Mark II,...,1,8.9,Etat,ENT-023,0,NaN,1.0,Aucune,0.0,fonctionnel
9127,PE-000759,2025-11-02,Borgou,Parakou,9.91264,3.02302,NaN,1995,Forage equipe PMH,India Mark II,...,1,8.9,Etat,ENT-023,0,NaN,1.0,Aucune,0.0,fonctionnel
5840,PE-001074,2025-09-11,Zou,Bohicon,7.08139,2.53166,183.4,2006,Puits traditionnel ameliore,Aucune (puisage manuel),...,0,53.4,Etat,ENT-023,1,8.0,1.0,Aucune,0.0,fonctionnel
2984,PE-001074,2025-09-11,Zou,Bohicon,7.08139,2.53166,183.4,2006,Puits traditionnel ameliore,Aucune (puisage manuel),...,0,53.4,Etat,ENT-023,1,8.0,1.0,Aucune,0.0,fonctionnel


In [62]:
n_ids_hors_doublons_exacts = df.drop_duplicates()["id_point_eau"].duplicated().sum()
print(f"id_parcelle encore répétés après suppression des doublons exacts : "
      f"{n_ids_hors_doublons_exacts}")

id_parcelle encore répétés après suppression des doublons exacts : 7


In [63]:
avant = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"{avant} -> {len(df)} lignes ({avant - len(df)} supprimees)")

print("\nRepartition de la cible :")
print(df["etat_fonctionnement"].value_counts(normalize=True).mul(100).round(1).to_string())


11472 -> 11407 lignes (65 supprimees)

Repartition de la cible :
etat_fonctionnement
fonctionnel              54.8
en panne                 37.2
fonctionnel a reparer     8.0


# A3. Les valeurs sentinelles

In [64]:
VALEURS_MAGIQUES = {-1, 0, -9, -99, -999, -9999, 999, 9999, 99999, -888, 888}

def detecter_sentinelles(df, cible="rendement_t_ha", seuil_trou=3):
    lignes = []
    for c in df.select_dtypes(include=[np.number]).columns.drop(cible, errors="ignore"):
        s = df[c].dropna()
        u = np.sort(s.unique())
        if len(u) < 3:
            continue
        iqr = s.quantile(0.75) - s.quantile(0.25) or s.std()
        for v, voisin in [(u[0], u[1]), (u[-1], u[-2])]:
            trou = abs(v - voisin) / iqr
            motifs = []
            if v in VALEURS_MAGIQUES:  motifs.append("nombre magique")
            if v < 0:                  motifs.append("negatif impossible")
            if trou > seuil_trou:      motifs.append("valeur isolee")
            if motifs:
                lignes.append({"colonne": c, "valeur": v, "n": int((s == v).sum()),
                               "trou_en_IQR": round(trou, 1), "motifs": ", ".join(motifs)})
    return pd.DataFrame(lignes)

display(detecter_sentinelles(df))

,colonne,valeur,n,trou_en_IQR,motifs
0,latitude,0.000000,214,1.8,nombre magique
1,longitude,0.000000,214,0.7,nombre magique
2,annee_construction,0.000000,168,90.1,"nombre magique, valeur isolee"
3,niveau_statique_m,7451.525683,1,370.8,valeur isolee
4,debit_essai_m3_h,-1.000000,62,0.5,"nombre magique, negatif impossible"
5,population_desservie,0.000000,95,0.1,nombre magique
6,cotisation_mensuelle_fcfa,0.000000,3006,0.1,nombre magique
7,nb_pannes_12_mois,0.000000,3379,0.5,nombre magique
8,mois_depuis_derniere_maintenance,0.000000,14,0.1,nombre magique
9,nb_jours_arret_12_mois,0.000000,599,0.0,nombre magique


In [65]:
SENTINELLES = {

    "latitude": [-999],
    "longitude": [-999],
    "annee_construction": [-999],
    "population_desservie": [-999],
    "debit_essai_m3_h": [-999],
}

for col, valeurs in SENTINELLES.items():
    n = df[col].isin(valeurs).sum()
    df[col] = df[col].replace(valeurs, np.nan)
    print(f"{col:<24} {n:>4} sentinelle(s) -> NaN   "
          f"(manquants : {df[col].isna().sum()}, soit {df[col].isna().mean()*100:.1f} %)")


latitude                    0 sentinelle(s) -> NaN   (manquants : 0, soit 0.0 %)
longitude                   0 sentinelle(s) -> NaN   (manquants : 0, soit 0.0 %)
annee_construction          0 sentinelle(s) -> NaN   (manquants : 0, soit 0.0 %)
population_desservie        0 sentinelle(s) -> NaN   (manquants : 0, soit 0.0 %)
debit_essai_m3_h            0 sentinelle(s) -> NaN   (manquants : 1053, soit 9.2 %)


In [66]:
print(df[[    "latitude",
    "longitude",
    "annee_construction",
    "population_desservie",
    "debit_essai_m3_h"]].describe().round(1))
print()
print("Sentinelles restantes :", len(detecter_sentinelles(df)))

       latitude  longitude  annee_construction  population_desservie  \
count   11407.0    11407.0             11407.0               11407.0   
mean        8.0        2.2              1973.7                 440.1   
std         2.0        0.6               241.6                 298.2   
min         0.0        0.0                 0.0                   0.0   
25%         6.7        1.8              1992.0                 225.0   
50%         7.3        2.2              2003.0                 373.0   
75%         9.7        2.6              2014.0                 586.0   
max        12.3        3.9              2024.0                3222.0   

       debit_essai_m3_h  
count           10354.0  
mean                3.1  
std                 2.1  
min                -1.0  
25%                 1.6  
50%                 2.7  
75%                 4.2  
max                16.8  

Sentinelles restantes : 11


# A4. Erreur d'unité

In [67]:
print(df["profondeur_forage_m"].describe())

mask_cm = df["profondeur_forage_m"] > 100

print("Nombre de valeurs à convertir :", mask_cm.sum())

df.loc[mask_cm, "profondeur_forage_m"] /= 100

count    11407.000000
mean        98.023898
std        558.376915
min          5.000000
25%         31.200000
50%         53.200000
75%         71.600000
max      11050.000000
Name: profondeur_forage_m, dtype: float64
Nombre de valeurs à convertir : 458


# A5. Incohérence physique

In [68]:
mask_incoherent = (
    df["niveau_statique_m"] > df["profondeur_forage_m"]
)

print("Nombre de lignes incohérentes :", mask_incoherent.sum())

Nombre de lignes incohérentes : 372


In [69]:
df.loc[
    mask_incoherent,
    ["niveau_statique_m", "profondeur_forage_m"]
]

,niveau_statique_m,profondeur_forage_m
35,31.848397,5.000
93,72.300000,1.048
111,19.281303,5.000
114,31.300000,1.368
146,9.900000,1.057
...,...,...
11248,67.900000,1.097
11250,41.600000,1.032
11282,22.300000,1.001
11378,17.000000,1.008


In [70]:
df.loc[
    mask_incoherent,
    "niveau_statique_m"
].describe()

count     372.000000
mean       66.529932
std       384.710701
min         5.400000
25%        27.950000
50%        43.050718
75%        62.719960
max      7451.525683
Name: niveau_statique_m, dtype: float64

In [71]:
df.loc[
    mask_incoherent,
    ["niveau_statique_m", "profondeur_forage_m"]
].assign(
    ecart=lambda x:
        x["niveau_statique_m"] - x["profondeur_forage_m"]
)

,niveau_statique_m,profondeur_forage_m,ecart
35,31.848397,5.000,26.848397
93,72.300000,1.048,71.252000
111,19.281303,5.000,14.281303
114,31.300000,1.368,29.932000
146,9.900000,1.057,8.843000
...,...,...,...
11248,67.900000,1.097,66.803000
11250,41.600000,1.032,40.568000
11282,22.300000,1.001,21.299000
11378,17.000000,1.008,15.992000


In [72]:
df.loc[
    mask_incoherent,
    "niveau_statique_m"
] = np.nan

# A6. Harmonisez les libellés

In [73]:
COLS_CAT = [c for c in df.select_dtypes(  include=["object", "string", "category"]).columns if c != "id_point_eau"]
for colonne in COLS_CAT:
    print("\n", )
    display(df[colonne].value_counts())

date_releve
2025-03-22    50
2025-07-11    50
2025-10-12    49
2025-04-12    49
2025-01-16    48
              ..
01/07/2025     1
01/08/2025     1
17/03/2025     1
14/05/2025     1
06/08/2025     1
Name: count, Length: 533, dtype: int64

departement
Atlantique      1383
Zou             1380
Borgou          1341
Oueme           1204
Atacora         1072
Collines         976
Couffo           846
Alibori          785
Plateau          763
Mono             753
Donga            644
  Zou             40
  Borgou          34
  Atacora         30
  Oueme           29
  Atlantique      26
  Donga           19
  Mono            18
  Collines        18
  Couffo          17
  Alibori         16
  Plateau         13
Name: count, dtype: int64

commune
So-Ava          188
N'Dali          184
Agbangnizoun    184
Parakou         183
Toffo           180
               ... 
ZOGBODOMEY        2
BOHICON           2
PERERE            2
GOGOUNOU          2
KLOUEKANME        1
Name: count, Length: 152, dtype: int64

type_ouvrage
Forage equipe PMH              5029
Puits moderne                  2171
Adduction d'eau villageoise    1725
Poste d'eau autonome           1612
Puits traditionnel ameliore     870
Name: count, dtype: int64

type_pompe
India Mark II              3152
Aucune (puisage manuel)    2202
Immergee solaire           1919
Vergnet                    1467
Immergee electrique         910
Volanta                     708
Kardia                      506
Immergee thermique          484
india mark 2                 35
Immergee Solaire             24
Name: count, dtype: int64

...

mode_paiement
Au volume              4311
Cotisation annuelle    3497
Gratuit                3080
Forfait mensuel         519
Name: count, dtype: int64

maitre_ouvrage
Commune                    3173
Etat                       2573
ONG internationale         2148
Cooperation bilaterale     1790
Association villageoise    1024
Prive                       699
Name: count, dtype: int64

installateur
ENT-011    251
ENT-041    251
ENT-046    240
ENT-044    239
ENT-050    238
ENT-007    235
ENT-016    234
ENT-018    234
ENT-035    233
ENT-052    233
ENT-014    233
ENT-032    232
ENT-042    229
ENT-036    228
ENT-051    227
ENT-028    226
ENT-026    226
ENT-045    226
ENT-027    225
ENT-008    225
ENT-021    225
ENT-031    224
ENT-023    224
ENT-029    222
ENT-048    221
ENT-034    221
ENT-017    220
ENT-022    220
ENT-033    218
ENT-002    217
ENT-040    215
ENT-038    215
ENT-024    215
ENT-043    212
ENT-039    212
ENT-020    211
ENT-037    210
ENT-019    210
ENT-001    209
ENT-003    209
ENT-012    208
ENT-013    207
ENT-009    206
ENT-025    205
ENT-010    203
ENT-049    202
ENT-005    201
ENT-004    201
ENT-030    201
ENT-015    197
ENT-047    195
ENT-006    186
Name: count, dtype: int64

intervention_prevue
Aucune                    6254
Rehabilitation lourde     2085
Remplacement de pompe     1750
Reparation legere          523
Abandon                    407
Remplacement de pieces     388
Name: count, dtype: int64

etat_fonctionnement
fonctionnel              6254
en panne                 4242
fonctionnel a reparer     911
Name: count, dtype: int64

In [74]:
SYNONYMES = {
    "type_pompe": {"India mark ii": "India mark 2"}
}
def normaliser(s):
    return (s.str.strip()
             .str.replace(r"\s+", " ", regex=True)
             .str.normalize("NFKD")
             .str.encode("ascii", "ignore").str.decode("utf-8")
             .str.capitalize())

for col in COLS_CAT:
    if col in SYNONYMES:
        df[col] = df[col].replace(SYNONYMES[col])

for col in COLS_CAT:
    df[col] = normaliser(df[col])




parsing des dates

In [75]:
brut = df["date_releve"].astype(str)
est_fr = brut.str.match(r"^\d{2}/\d{2}/\d{4}$")
p1 = brut[est_fr].str.slice(0, 2).astype(int)
p2 = brut[est_fr].str.slice(3, 5).astype(int)
print(f"ISO {(~est_fr).sum()} | FR {est_fr.sum()}")
print(f"position1 > 12 : {(p1>12).sum()}  position2 > 12 : {(p2>12).sum()}  ambigues : {((p1<=12)&(p2<=12)).sum()}")


df["date_releve"] = pd.to_datetime(df["date_releve"], format="mixed", dayfirst=True)
assert df["date_releve"].isna().sum() == 0

df["annee_releve"]     = df["date_releve"].dt.year
df["mois_releve"]      = df["date_releve"].dt.month
df["trimestre_releve"] = df["date_releve"].dt.quarter
df["periode_releve"]   = df["date_releve"].dt.to_period("Q").astype(str)

print(df["date_releve"].min().date(), "->", df["date_releve"].max().date())
print(df["annee_releve"].value_counts().sort_index().to_string())

ISO 11067 | FR 340
position1 > 12 : 195  position2 > 12 : 0  ambigues : 145
2025-01-02 -> 2025-12-11
annee_releve
2025    11407


In [76]:
for colonne in COLS_CAT:
    print("\n", )
    print(df[colonne].value_counts())



date_releve
2025-03-22    52
2025-01-16    50
2025-11-07    50
2025-12-10    49
2025-12-04    49
              ..
2025-07-12     2
2025-10-12     2
2025-03-12     1
2025-05-01     1
2025-04-12     1
Name: count, Length: 330, dtype: int64


departement
Zou           1420
Atlantique    1409
Borgou        1375
Oueme         1233
Atacora       1102
Collines       994
Couffo         863
Alibori        801
Plateau        776
Mono           771
Donga          663
Name: count, dtype: int64


commune
So-ava          192
N'dali          191
Parakou         191
Agbangnizoun    189
Toffo           185
               ... 
Toucountouna    116
Natitingou      114
Athieme         112
Gogounou        106
Kerou           104
Name: count, Length: 76, dtype: int64


type_ouvrage
Forage equipe pmh              5029
Puits moderne                  2171
Adduction d'eau villageoise    1725
Poste d'eau autonome           1612
Puits traditionnel ameliore     870
Name: count, dtype: int64


type_pompe
India mar

# A7. Etat des valeurs manquants

In [83]:

df["coordonnees_manquantes"] = (
    df["latitude"].isna() |
    df["longitude"].isna()
).astype(int)

# Indicateur année inconnue
df["annee_inconnue"] = (
    df["annee_construction"].isna()
).astype(int)

In [84]:

print(
    df.groupby("coordonnees_manquantes")["nb_pannes_12_mois"]
    .mean()
    .mul(100)
)

coordonnees_manquantes
0    137.415622
Name: nb_pannes_12_mois, dtype: float64


In [82]:
# Taux de panne selon l'année
print(
    df.groupby("annee_inconnue")["nb_pannes_12_mois"]
    .mean()
    .mul(100)
)

annee_inconnue
0    137.415622
Name: nb_pannes_12_mois, dtype: float64
